# 08 · Final research report

Integrated analysis: objectives, data, methods, experimental design, metrics, results, error analysis, limitations and conclusions. All numbers are computed from the versioned result files; preliminary (AI-drafted, unreviewed) evaluation data are labelled as such. Writes `reports/research_summary.md` and `reports/abstract_draft.md`.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

## 1. Research objectives

Determine how well a multilingual retrieval-augmented assistant finds the approved university / government source that answers an international student's question asked in Korean, English, Indonesian, Vietnamese or Chinese, and how much the improved hybrid retrieval contributes relative to vector-only and equal-weight hybrid baselines; measure what can be measured of the generated answers.

In [ ]:
stats = json.loads((A.SNAP / "dataset_statistics.json").read_text())
ch = pd.read_csv(A.SNAP / "chunk_metrics.csv"); active = ch[ch["active"]]
cfg, gen = EXPERIMENTS["shared"], EXPERIMENTS["generation"]
summary_rows = [("Active documents", stats["documents"]["active_searchable"]),
                ("Original languages", ", ".join(f"{k} {v}" for k, v in stats["documents"]["by_language"].items())),
                ("Pages (PDF metadata only)", stats["pages"]["total_pages_from_extraction_metadata"]),
                ("Active chunks / embedded", f"{stats['chunks']['active']} / {stats['chunks']['active_with_embedding']}"),
                ("Chunk size, characters: median (IQR)", f"{active['chars'].median():.0f} ({active['chars'].quantile(.25):.0f}-{active['chars'].quantile(.75):.0f})"),
                ("Embedding", f"{cfg['embedding_model']} ({cfg['embedding_variant']}), {cfg['embedding_dimensions']} dimensions"),
                ("Vector index", f"{cfg['vector_store']['index']}; pgvector {stats['pgvector_version']}"),
                ("Chunker", f"target {cfg['chunker']['target_tokens']} / max {cfg['chunker']['max_tokens']} / overlap {cfg['chunker']['overlap_tokens']} est. tokens"),
                ("Generation", f"{gen['provider']}; prompt {gen['prompt_version']} ({gen['prompt_sha256_12']})")]
dataset_table = pd.DataFrame(summary_rows, columns=["item", "value"])
dataset_table.to_csv(A.REPORTS / "dataset_summary.csv", index=False); dataset_table

## 2-8. Data, preprocessing, chunking, embedding, vector database, retrieval and generation

- **Data**: approved documents of the SCH International Student Hub knowledge base (university notices and guides, Korean government guidance), English and Korean originals; only the current approved version of each document is searchable (`searchable_chunks`).
- **Preprocessing**: extraction from web pages, PDF (PyMuPDF, page markers), DOCX and pasted notices; text normalisation; human review and approval before indexing.
- **Chunking**: heading/paragraph-aware chunker v1 with a sliding token overlap inside long sections (see notebook 02 for measured sizes and overlap).
- **Embedding**: multilingual-e5-small, ONNX int8, `passage:` / `query:` prefixes, 384 dimensions, CPU.
- **Vector database**: Supabase PostgreSQL + pgvector HNSW (cosine) with GIN full-text and trigram indexes.
- **Retrieval**: B0 vector-only; E1 equal-weight RRF of vector and keyword legs in SQL; E2 application-side weighted fusion with rarity-weighted, script-aware keyword coverage, visa-code normalisation, a multilingual glossary, code-mismatch demotion, de-duplication and a per-document cap; E2S adds the assistant's evidence selection. Reranking was not implemented.
- **Generation**: Gemini (primary + fallback) with a versioned system prompt; server-side citation, grounding and prompt-leak checks.

## 9-10. Experimental design and metrics

Two retrieval datasets of 100 synthetic questions each (20 scenarios × 5 languages): the **pilot v0.1** set was used during development (tuning) and is reported as a development set; the **held-out v0.2** set was frozen before its first evaluation and never used for tuning. Both were drafted by an AI assistant and are **not yet human-reviewed**; results are preliminary. Document-level Recall@1, Recall@5, MRR and nDCG@5 with 95% cluster-bootstrap intervals over scenarios; paired differences in percentage points. Two targets: the local working tree (in-memory index rebuilt with the production chunker and embedder) and the deployed production API (real pgvector index).

In [ ]:
frames = {}
for target in ("local", "production"):
    for ds in ("heldout_v0.2", "pilot_v0.1"):
        f = A.RESULTS / f"retrieval_{target}_{ds}.csv"
        if target == "production":   # erratum 2026-11-23: announcement ids corrected (scripts/rescore_production.py)
            f = A.RESULTS / f"retrieval_{target}_{ds}_rescored.csv"
        if f.exists():
            frames[(target, ds)] = A.load_retrieval(f.name)
avail = pd.DataFrame([(t, d, len(df), df["method"].nunique(), int((df["error"].fillna("") != "").sum()))
                      for (t, d), df in frames.items()], columns=["target", "dataset", "rows", "methods", "failed requests"])
avail

## 11-12. Results

In [ ]:
order = ["B0", "E1", "E2", "E2S"]
res = []
for (t, d), df in frames.items():
    s = A.summarize(df, ["method"]); s.insert(0, "dataset", d); s.insert(0, "target", t); res.append(s)
res = pd.concat(res, ignore_index=True)
res["method"] = pd.Categorical(res["method"], order, ordered=True)
res = res.sort_values(["target", "dataset", "method"])
def ci(r, m): return f"{r[m]:.3f} [{r[m+'_lo']:.3f}, {r[m+'_hi']:.3f}]"
table = pd.DataFrame([dict(target=r.target, dataset=r.dataset, method=f"{r.method} {A.METHODS[r.method].split(' ', 1)[1]}",
                           n=r.n, **{"Recall@1": ci(r, "r1"), "Recall@5": ci(r, "r5"), "MRR": ci(r, "rr"),
                                     "nDCG@5": ci(r, "ndcg5")}) for _, r in res.iterrows()])
table.to_csv(A.REPORTS / "retrieval_results_table.csv", index=False); table

In [ ]:
heldout = res[res.dataset == "heldout_v0.2"]
fig, ax = plt.subplots(figsize=(10, 4))
targets = [t for t in ("local", "production") if t in set(heldout["target"])]
tab = heldout.assign(group=heldout["target"].str.capitalize()).set_index(["method", "group"])
A.grouped_bars(ax, tab, [t.capitalize() for t in targets], [m for m in order if m in set(heldout["method"])], "rr")
ax.set_ylabel("MRR (95% CI)"); ax.set_title(f"MRR on the held-out set (n = {int(heldout['n'].iloc[0])} questions, 20 scenarios, 5 languages)")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1)); fig.tight_layout(); print(A.save(fig, "08_mrr_heldout_summary"))

In [ ]:
pairs = []
for (t, d), df in frames.items():
    for a, b in (("E2", "B0"), ("E2", "E1")):
        if {a, b} <= set(df["method"]):
            for col in ("r1", "rr"):
                r = A.paired_difference(df, a, b, col); r.update(target=t, dataset=d); pairs.append(r)
pairs = pd.DataFrame(pairs)
pairs["result"] = pairs.apply(lambda r: f"{100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]"
                              + (" *" if r["ci_excludes_zero"] else ""), axis=1)
pairs[["target", "dataset", "a", "b", "metric", "n", "result", "mcnemar_p"]].round(3)

`*` = the 95% interval excludes 0.

## 13. Error analysis

In [ ]:
key = ("production", "heldout_v0.2") if ("production", "heldout_v0.2") in frames else ("local", "heldout_v0.2")
df = frames[key]
bl = A.summarize(df[df.method == "E2"], ["language"], metrics=("r1", "rr")).set_index("language").loc[A.LANGS]
display(bl[["n", "r1", "r1_lo", "r1_hi", "rr"]].round(3))
ans_f = A.RESULTS / "answers_production_heldout_v0.2.jsonl"
answers = pd.read_json(ans_f, lines=True) if ans_f.exists() else pd.DataFrame()
gen_rate = (answers["answer_generation"] == "llm").mean() if len(answers) else np.nan
served = sorted(set(answers.loc[answers.get("answer_generation", pd.Series(dtype=str)) == "llm", "model"].dropna())) if len(answers) else []
print("answer cases:", len(answers), "| generation success:", None if np.isnan(gen_rate) else round(gen_rate, 3), "| served models:", served)

## 14-15. Limitations, conclusions and generated reports

In [ ]:
def get(t, d, a, b, m):
    r = pairs[(pairs.target == t) & (pairs.dataset == d) & (pairs.a == a) & (pairs.b == b) & (pairs.metric == m)]
    return None if r.empty else r.iloc[0]
def val(t, d, m, k="r1"):
    r = res[(res.target == t) & (res.dataset == d) & (res.method == m)]
    return None if r.empty else r.iloc[0]
main_t = "production" if ("production", "heldout_v0.2") in frames else "local"
e2b0 = get(main_t, "heldout_v0.2", "E2", "B0", "rr"); e2e1 = get(main_t, "heldout_v0.2", "E2", "E1", "rr")
v = {m: val(main_t, "heldout_v0.2", m) for m in ("B0", "E1", "E2")}
pilot = get("local", "pilot_v0.1", "E2", "B0", "r1"); held = get("local", "heldout_v0.2", "E2", "B0", "r1")
def pp(r): return f"{100*r['diff']:+.1f} pp (95% CI {100*r['lo']:+.1f} to {100*r['hi']:+.1f})"
answer_line = ("Answer quality (correctness, relevance, faithfulness) was not evaluated: no human-reviewed scores exist."
               + (f" In {len(answers)} recorded production cases, {gen_rate:.0%} produced a generated answer"
                  f" (served by {', '.join(served) or 'no model'})." if len(answers) else ""))
st = stats
abstract = f"""**Draft abstract (preliminary - evaluation questions and labels are AI-drafted and not yet human-reviewed).**

International students need accurate answers about immigration, residence and university procedures in their own
language, while the authoritative sources are written mostly in Korean and English. We describe and evaluate a
retrieval-augmented assistant for international students at Soonchunhyang University built on
{st['documents']['active_searchable']} approved documents ({st['chunks']['active']} chunks; {cfg['embedding_model']},
{cfg['embedding_dimensions']}-dimensional embeddings in PostgreSQL/pgvector). On a held-out set of 100 synthetic questions
(20 scenarios in Korean, English, Indonesian, Vietnamese and Chinese) evaluated against the deployed system, the improved
hybrid retriever reached MRR {v['E2']['rr']:.3f} versus {v['B0']['rr']:.3f} for vector-only retrieval and
{v['E1']['rr']:.3f} for equal-weight hybrid fusion (paired differences {pp(e2b0)} and {pp(e2e1)}). In the local
evaluation, the Recall@1 gain over vector-only retrieval on the development set used for tuning ({pp(pilot)}) was larger
than on the held-out set ({pp(held)}), illustrating the optimism of tuning-set evaluation. {answer_line} We release the evaluation framework, a review workflow for
independent labels and the statistics of the knowledge base."""
display(Markdown(abstract))
(A.REPORTS / "abstract_draft.md").write_text(abstract + "\n")
limits = """- Questions, translations and relevance labels are AI-drafted and **not human-reviewed**; results are preliminary.
- 20 scenarios per dataset: intervals are wide; per-language differences of one question equal 5 percentage points.
- The pilot set was used for tuning (development set); only held-out results should be reported as estimates.
- Relevance is binary and document-level; partial or passage-level relevance is not measured.
- The local target rebuilds the index from the public corpus snapshot; production includes network latency.
- Reranking was not implemented; query rewriting (LLM) was not part of the retrieval experiments.
- Answer-quality metrics requiring judgement are not evaluated; the primary Gemini model produced no successful answers in
  the recorded production runs, so no model comparison is possible.
- Knowledge-base validity metadata is sparse (see notebook 01); time-bound documents can be out of date."""
summary = ["# SCH International Student Hub - multilingual RAG evaluation (preliminary)", "",
           f"Generated by notebook 08 from versioned results; statistics as of {st['generated_at'][:10]}.", "",
           "## Dataset", dataset_table.to_markdown(index=False), "", "## Retrieval results", table.to_markdown(index=False), "",
           "`[a, b]` = 95% cluster-bootstrap interval over scenarios.", "", "## Paired differences",
           pairs[["target", "dataset", "a", "b", "metric", "n", "result"]].to_markdown(index=False), "", "`*` = interval excludes 0.", "",
           "## Answer generation", answer_line, "", "## Limitations", limits, "", "## Abstract draft", abstract]
(A.REPORTS / "research_summary.md").write_text("\n".join(summary) + "\n")
display(Markdown("### Limitations\n" + limits)); print("wrote reports/research_summary.md and reports/abstract_draft.md")